# Deep FLASH medial temporal lobe parcellation

Segment hippocampal and entorhinal regions with the pretrained T1 model.
**Resources:** downloads a sample T1, templates, priors and model weights on
first use. This 3-D workflow can require substantial CPU time and memory.
Optional T2 input must already be aligned to T1.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import matplotlib.pyplot as plt
import ants
import antstorch

device = "cpu"
antstorch.set_default_device(device)

## Load the sample MRI or choose a local image

In [ ]:
from pathlib import Path

# Set an absolute path to use your own scalar 3-D T1 MRI instead.
t1_path = None
if t1_path is None:
    t1_path = antstorch.get_antstorch_data("mprage_hippmapp3r")
t1 = ants.image_read(str(t1_path)).clone("float")
assert t1.dimension == 3 and not t1.has_components
assert np.isfinite(t1.numpy()).all()
print("Input:", t1_path, "shape:", t1.shape, "spacing:", t1.spacing)

In [ ]:
# Supply an absolute path for optional T2; None selects the T1-only model.
t2_path = None
t2 = None if t2_path is None else ants.image_read(str(t2_path)).clone("float")
if t2 is not None:
    assert t2.dimension == 3 and not t2.has_components
    assert t2.shape == t1.shape
    for attribute in ("spacing", "origin", "direction"):
        np.testing.assert_allclose(getattr(t2, attribute), getattr(t1, attribute))
    # Matching metadata does not prove anatomical alignment: inspect it too.
    ants.plot(t1, overlay=t2, axis=2, nslices=6, overlay_alpha=0.4)
result = antstorch.deep_flash(
    t1, t2=t2, do_preprocessing=True, use_rank_intensity=True,
    backend="antstorch", device=device, verbose=True,
)
segmentation = result["segmentation_image"]
probabilities = result["probability_images"]
assert segmentation.shape == t1.shape
assert np.isin(segmentation.numpy(), [0, *range(5, 19)]).all()
for attribute in ("spacing", "origin", "direction"):
    np.testing.assert_allclose(getattr(segmentation, attribute), getattr(t1, attribute))
for probability in probabilities:
    assert probability.shape == t1.shape
    assert np.isfinite(probability.numpy()).all()

## Inspect fine anatomical regions
Labels 5–18 alternate left/right regions: aLEC, pMEC, perirhinal,
parahippocampal, DG/CA2/CA3/CA4, CA1 and subiculum. Background is 0.
Whole-head slices can hide these small structures, so inspect the hippocampal
foreground map as well as the discrete labels and zoom into relevant slices.

In [ ]:
ants.plot(t1, overlay=segmentation, axis=1, nslices=8,
          overlay_cmap="jet", overlay_alpha=0.6)
hippocampus = result["hippocampal_probability_image"]
assert hippocampus.shape == t1.shape and np.isfinite(hippocampus.numpy()).all()
ants.plot(t1, overlay=hippocampus, axis=1, nslices=8,
          overlay_cmap="hot", overlay_alpha=0.6)
output_dir = Path("example_outputs/deep_flash")
output_dir.mkdir(parents=True, exist_ok=True)
ants.image_write(segmentation, str(output_dir / "labels.nii.gz"))
ants.image_write(hippocampus, str(output_dir / "hippocampal_probability.nii.gz"))
voxel_volume = float(np.prod(t1.spacing))
for label in range(5, 19):
    volume = np.count_nonzero(segmentation.numpy() == label) * voxel_volume
    print(f"Label {label}: {volume:.2f} cubic physical units")
print("Outputs (matching files are overwritten):", output_dir.resolve())

These checks validate output structure, not anatomical accuracy or agreement
with ANTsPyNet. Inspect left/right orientation and individual subregions before
interpreting regional volume differences.